# Genetic Algorithm(neuroevolution) for Unit Placement

## Импорт необходимых библиотек и подготовка среды

In [26]:
import torch
import torch.nn as nn
import numpy as np
import copy
import random
import matplotlib.pyplot as plt

POP_SIZE = 15          # Размер популяции (сколько ботов в одном поколении)
MUTATION_RATE = 0.05   # Вероятность мутации весов (5%)
INPUT_SIZE = 102       # 100 клеток поля + бюджет + раунд
HIDDEN1 = 64           # Размер первого скрытого слоя
HIDDEN2 = 32           # Размер второго скрытого слоя
OUTPUT_SIZE = 225      # 9 комбинаций юнитов * 25 клеток

HOST = '127.0.0.1'     # Локальный адрес для связи с Unity
PORT = 5005            # Порт

## Нейронная сеть

In [21]:
class BattleNet(nn.Module):
    def __init__(self, input_size=102, hidden1 = 64, hidden2 = 32, output_size = 225):
        super().__init__()
        self.fc1 = nn.Linear(input_size,hidden1)
        self.fc2 = nn.Linear(hidden1,hidden2)
        self.fc3 = nn.Linear(hidden2,output_size)
        
        for param in self.parameters():
            param.requires_grad = False
    
    def forward(self, x):
        x = torch.FloatTensor(x)
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)
        x = torch.relu(x)
        x = self.fc3(x)
        
        return torch.sigmoid(x)
    
    def get_weights(self):
        weights = []
        for param in self.parameters():
            weights.extend(param.numpy().flatten())
        return np.array(weights)
    
    def set_weights(self, weights):
        """Принимает список чисел (от ГА) и вставляет их обратно в структуру нейросети"""
        start = 0
        for param in self.parameters():
            # Считаем, сколько чисел нужно для конкретного слоя
            size = param.numel()
            # Берем нужный кусок из длинного списка и меняем его форму под слой
            new_param = weights[start:start+size].reshape(param.shape)
            param.copy_(torch.from_numpy(new_param))
            start += size

В данном разделе определяется структура **полносвязной нейронной сети** (Multi-Layer Perceptron, MLP). Сеть обучается методом **нейроэволюции**, поэтому стандартный механизм обратного распространения ошибки (*backpropagation*) полностью отключен.

### 1. Математическая модель

Сеть выполняет последовательное нелинейное преобразование входного вектора состояния среды $\mathbf{x}$ в выходной вектор предиктов $\mathbf{y}$ через систему скрытых слоев.

#### **Структура входных данных**


Входной вектор $\mathbf{x} \in \mathbb{R}^{102}$ формируется путем **объединения** признаков среды:
$$\mathbf{x} = [s_1, s_2, \dots, s_{100}, b, r]$$
Где:
* $s_{1 \dots 100}$ — состояние клеток игрового поля ($0$ — пусто, $1$ — препятствие);
* $b$ — текущий бюджет эликсира;
* $r$ — порядковый номер игрового раунда.


#### **Процесс прямого прохода (Forward Pass)**

Вычисления в слоях производятся по следующим формулам:

1. **Первый скрытый слой ($\mathbf{h_1}$):**
   $$\mathbf{h_1} = \text{ReLU}(W_1 \mathbf{x} + \mathbf{b_1})$$
   Где $W_1$ — матрица весов размера $64 \times 102$, а $\mathbf{b_1}$ — вектор смещений (bias).

2. **Второй скрытый слой ($\mathbf{h_2}$):**
   $$\mathbf{h_2} = \text{ReLU}(W_2 \mathbf{h_1} + \mathbf{b_2})$$
   Где $W_2$ — матрица весов размера $32 \times 64$.

3. **Выходной слой ($\mathbf{y}$):**
   $$\mathbf{y} = \sigma(W_3 \mathbf{h_2} + \mathbf{b_3})$$
   Используется логистическая функция (сигмоида): $\sigma(z) = \frac{1}{1 + e^{-z}}$, которая масштабирует значения в диапазон $(0, 1)$, интерпретируемый как «уверенность» сети в выборе позиции.

### 2. Особенности реализации для генетического алгоритма


* **Заморозка градиентов:** Так как обучение происходит через эволюцию весов, свойство `requires_grad` для всех параметров установлено в `False`. Это отключает механизм автоматического дифференцирования PyTorch, что существенно снижает нагрузку на оперативную память и ускоряет выполнение итераций:
  $$\frac{\partial \mathcal{L}}{\partial \theta} = 0$$

* **Генотип особи:** Полный набор весов и смещений сети $\theta = \{W, \mathbf{b}\}$ извлекается в виде одномерного массива. Это позволяет операторам мутации и кроссовера напрямую корректировать структуру связей агента, изменяя его поведение без использования градиентных методов.

* **Функция активации ReLU:** Использование $f(z) = \max(0, z)$ в скрытых слоях обеспечивает избирательную активацию нейронов, помогая сети игнорировать малозначимые входные данные (например, пустые клетки поля).

### 3. Техническая спецификация

| Слой | Тип | Размерность | Функция активации |
| :--- | :--- | :--- | :--- |
| **Input** | Linear | $102 \to 64$ | ReLU |
| **Hidden** | Linear | $64 \to 32$ | ReLU |
| **Output** | Linear | $32 \to 225$ | Sigmoid |

### Декодер

In [22]:
def decode_output(probabilities, budget):
    """
    probabilities: 225 чисел от нейронки
    budget: сколько эликсира дала Unity
    """
    # Стоимость: [тип][уровень]
    # 1: Рыцарь, 2: Лучник, 3: Маг
    UNIT_COSTS = {
        1: {1: 3, 2: 6, 3: 9},
        2: {1: 2, 2: 4, 3: 6},
        3: {1: 4, 2: 8, 3: 12}
    }

    layout = ["0:0"] * 25
    current_budget = budget
    occupied_cells = set()

    # 1. Группируем выходы: каждые 25 нейронов — это конкретный юнит+уровень
    # Нам нужно превратить 225 чисел в список понятных вариантов
    all_options = []
    
    # Всего 9 комбинаций (3 типа * 3 уровня)
    combinations = [
        (1, 1), (1, 2), (1, 3), # Рыцари 1,2,3
        (2, 1), (2, 2), (2, 3), # Лучники 1,2,3
        (3, 1), (3, 2), (3, 3)  # Маги 1,2,3
    ]

    for combo_idx, (u_type, u_level) in enumerate(combinations):
        start_idx = combo_idx * 25
        # Берем 25 чисел, отвечающих за эту конкретную комбинацию
        chunk = probabilities[start_idx : start_idx + 25]
        
        for cell_idx, confidence in enumerate(chunk):
            all_options.append({
                'cell': cell_idx,
                'type': u_type,
                'level': u_level,
                'conf': confidence,
                'cost': UNIT_COSTS[u_type][u_level]
            })

    # 2. Сортируем все 225 вариантов по уверенности (самые желанные — в начало)
    all_options.sort(key=lambda x: x['conf'], reverse=True)

    # 3. "Жадный" выбор: ставим, если есть место и деньги
    for opt in all_options:
        if opt['cell'] in occupied_cells:
            continue
        
        if opt['conf'] < 0.2: # Порог "осмысленности"
            continue

        if current_budget >= opt['cost']:
            layout[opt['cell']] = f"{opt['type']}:{opt['level']}"
            current_budget -= opt['cost']
            occupied_cells.add(opt['cell'])

    return ",".join(layout)


## Генетический алгоритм

In [23]:
class GAManager:
    def __init__(self, pop_size=15, mutation_power=0.05):
        self.pop_size = pop_size
        self.mutation_power = mutation_power # Насколько сильно меняются веса
        # Создаем популяцию
        self.population = [BattleNet() for _ in range(pop_size)]
        self.fitness_history = []

    def evolve(self, fitness_scores):
        """
        Переход к новому поколению
        fitness_scores: список урона для каждой сети [150, 10, 300...]
        """
        # 1. Логируем статистику для графиков
        self.fitness_history.append({
            'max': max(fitness_scores),
            'avg': np.mean(fitness_scores)
        })

        # 2. Селекция (Tournament Selection)
        # Мы не просто убиваем всех, а даем шанс разным особям
        new_population = []
        
        # Элитизм: сохраняем лучшего без изменений
        best_idx = np.argmax(fitness_scores)
        new_population.append(self.population[best_idx])

        # 3. Заполняем остальную популяцию
        while len(new_population) < self.pop_size:
            # Скрещиваем двух случайных лидеров из ТОП-3
            parent1, parent2 = self._select_parents(fitness_scores)
            
            child_weights = self._crossover(parent1, parent2)
            child_weights = self._mutate(child_weights)
            
            child = BattleNet()
            child.set_weights(child_weights)
            new_population.append(child)
            
        self.population = new_population
    
    def _select_parents(self, fitness_scores):
        """Выбирает двух РАЗНЫХ родителей через турнир"""
        def tournament():
            subset_indices = random.sample(range(self.pop_size), 4)
            best_in_subset = max(subset_indices, key=lambda i: fitness_scores[i])
            return self.population[best_in_subset]
        
        parent1 = tournament()
        parent2 = tournament()
        
        # Если выбрали одну и ту же сеть, пробуем еще раз для второго родителя
        # Ограничим попытки (например, до 10), на случай если вся популяция стала одинаковой
        attempts = 0
        while parent1 is parent2 and attempts < 10:
            parent2 = tournament()
            attempts += 1
            
        return parent1, parent2
    
    def _crossover(self, p1, p2):
        """Смешиваем веса двух родителей"""
        w1 = p1.get_weights()
        w2 = p2.get_weights()
        mask = np.random.randint(0, 2, size=w1.shape).astype(bool)
        return np.where(mask, w1, w2)
    
    def _mutate(self, weights):
        """Добавляем случайную мутацию"""
        mutation = np.random.normal(0, self.mutation_power, size=weights.shape)
        return weights + mutation

Класс `GAManager` реализует логику искусственного отбора для оптимизации весовых коэффициентов нейронных сетей. Алгоритм имитирует процесс биологической эволюции, итерируя через поколения для поиска глобального максимума функции приспособленности (Fitness).

### 1. Компоненты эволюционного процесса

Для обеспечения стабильного роста результатов и сохранения разнообразия в популяции используются следующие механизмы:

* **Элитизм (Elitism):** Стратегия, при которой особь с максимальным показателем Fitness (чемпион) переходит в следующее поколение без изменений. Это гарантирует, что достигнутый рекорд не будет потерян из-за деструктивной мутации.
* **Турнирная селекция (Tournament Selection):** Метод выбора родителей, при котором из популяции случайно выбирается группа особей (в данном случае 4), и победителем становится лучшая из них. Это позволяет сохранять генетическое разнообразие и давать шанс менее приспособленным особям, что предотвращает преждевременную сходимость алгоритма.
* **Равномерное скрещивание (Uniform Crossover):** Процесс создания потомка, при котором каждый «ген» (весовой коэффициент) выбирается случайно от одного из двух родителей с равной вероятностью:
    $$w_{child} = \begin{cases} w_{parent1}, & p < 0.5 \\ w_{parent2}, & p \geq 0.5 \end{cases}$$
* **Нормальная мутация (Gaussian Mutation):** Внесение случайного шума в веса потомка. Мы используем нормальное распределение с центром в $0$ и стандартным отклонением, задаваемым параметром `mutation_power`:
    $$w_{new} = w_{old} + \mathcal{N}(0, \sigma^2)$$

### 2. Математическая оценка и мониторинг

После каждой итерации (поколения) алгоритм фиксирует ключевые метрики для анализа эффективности обучения:
* **Max Fitness:** Максимальный урон, нанесенный лучшей конфигурацией.
* **Average Fitness:** Средний урон по всей популяции, показывающий общий уровень «интеллекта» поколения.

### 3. Технические особенности реализации

| Метод | Функция | Описание |
| :--- | :--- | :--- |
| `evolve` | Цикл развития | Отвечает за логирование статистики и формирование нового поколения. |
| `_select_parents` | Турнир | Обеспечивает выбор двух уникальных родительских структур. |
| `_crossover` | Рекомбинация | Создает маску для смешивания весовых матриц родителей. |
| `_mutate` | Модификация | Применяет случайные правки к весам для поиска новых стратегий. |

## Сервер и отрисовка

In [27]:
def plot_live_evolution(history):
    if not history:
        print("Визуализация: История пока пуста...")
        return
    
    gen_idx = len(history) - 1
    max_val = history[-1]['max']
    avg_val = history[-1]['avg']

    # Явный текстовый лог, чтобы не было "тишины"
    print(f"\n=== ОБНОВЛЕНИЕ ГРАФИКА: ПОКОЛЕНИЕ {gen_idx} ===")
    print(f"Лучший результат (Max): {max_val:.2f}")
    print(f"Средний результат (Avg): {avg_val:.2f}\n")

    generations = range(len(history))
    max_fitness = [h['max'] for h in history]
    avg_fitness = [h['avg'] for h in history]

    plt.figure(figsize=(10, 5))
    plt.plot(generations, max_fitness, label='Max Fitness', color='#2ca02c', marker='o', markersize=4)
    plt.plot(generations, avg_fitness, label='Avg Fitness', color='#1f77b4', linestyle='--')
    
    plt.title(f'Прогресс обучения (Поколение {gen_idx})', fontsize=14)
    plt.xlabel('Поколение')
    plt.ylabel('Fitness')
    plt.legend(loc='upper left')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.show()

In [28]:
ga = GAManager(pop_size=POP_SIZE, mutation_power=MUTATION_RATE)

def run_evolution_cycle():
    generation = 0
    
    print(f"Сервер запущен на {HOST}:{PORT}. Ожидание подключений из Unity...")
    
    try:
        while True:
            fitness_scores = []
            
            # Цикл сбора данных от всей популяции (15 особей)
            for i in range(POP_SIZE):
                with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                    s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
                    s.bind((HOST, PORT))
                    s.listen(1)
                    
                    # Здесь Python замирает и ждет Unity
                    conn, addr = s.accept()
                    with conn:
                        try:
                            # 1. Прием данных (Бюджет|Поле)
                            raw_data = conn.recv(4096).decode().strip()
                            if not raw_data:
                                print(f"  [!] Бот {i}: Получены пустые данные")
                                fitness_scores.append(0.0)
                                continue
                            
                            parts = raw_data.split('|')
                            budget = float(parts[0])
                            state_vector = np.fromstring(parts[1], sep=',')
                            
                            # 2. Инференс (выбор решения нейросетью)
                            current_brain = ga.population[i]
                            with torch.no_grad():
                                state_tensor = torch.FloatTensor(state_vector)
                                probabilities = current_brain.forward(state_tensor).numpy()
                            
                            # 3. Отправка ответа в Unity (расстановка)
                            layout_response = decode_output(probabilities, budget)
                            conn.sendall(layout_response.encode())
                            
                            # 4. Прием Fitness (результат боя)
                            fitness_raw = conn.recv(1024).decode().strip()
                            score = float(fitness_raw)
                            fitness_scores.append(score)
                            
                            print(f"  [OK] Бот {i+1}/{POP_SIZE} обработан. Fitness: {score:.2f}")
                            
                        except Exception as e:
                            print(f"  [X] Ошибка на боте {i}: {e}")
                            fitness_scores.append(0.0)

            # Когда собрали все 15 результатов — проводим эволюцию
            print(f"\n--> Поколение {generation} завершено. Запуск эволюции...")
            
            if len(fitness_scores) == POP_SIZE:
                ga.evolve(fitness_scores)
                
                # Принудительная отрисовка
                plt.close('all')
                clear_output(wait=True)
                plot_live_evolution(ga.fitness_history)
                
                generation += 1
            else:
                print(f"Ошибка: Собрано только {len(fitness_scores)} результатов. Перезапуск цикла.")

    except KeyboardInterrupt:
        print("\nОбучение прервано пользователем.")
    except Exception as e:
        print(f"\nКритическая ошибка системы: {e}")

# Запуск всего процесса

# Запуск
run_evolution_cycle()

Сервер запущен на 127.0.0.1:5005. Ожидание подключений из Unity...
  [OK] Бот 1/15 обработан. Fitness: 0.00
  [OK] Бот 2/15 обработан. Fitness: -1.00
  [OK] Бот 3/15 обработан. Fitness: 32.00

Обучение прервано пользователем.
